# Datenvisualisierung Dashboard

## Plotten der Sinuskurve

### 1. Importe

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### 2. Matplotlib-Funktion

In [2]:
def mplplot(df, **kwargs):
    fig = df.plot().get_figure()
    plt.close(fig)
    return fig

### 3. Sinus-Plot

In [3]:
def sine(frequency=1.0, amplitude=1.0, n=200, view_fn=mplplot):
    xs = np.arange(n) / n * 20.0
    ys = amplitude * np.sin(frequency * xs)
    df = pd.DataFrame(dict(y=ys), index=xs)
    return view_fn(df, frequency=frequency, amplitude=amplitude, n=n)

In [4]:
import holoviews as hv
import hvplot.pandas
import panel as pn


tap = hv.streams.PointerX(x=0)

def hvplot(df, frequency, **kwargs):
    plot = df.hvplot(width=500, padding=(0, 0.1))
    tap.source = plot

    def unit_circle(x):
        cx = np.cos(x * frequency)
        sx = np.sin(x * frequency)
        circle = hv.Path(
            [hv.Ellipse(0, 0, 2), [(-1, 0), (1, 0)], [(0, -1), (0, 1)]]
        ).opts(color="black")
        triangle = hv.Path(
            [[(0, 0), (cx, sx)], [(0, 0), (cx, 0)], [(cx, 0), (cx, sx)]]
        ).opts(color="red", line_width=2)
        labels = hv.Labels(
            [(cx / 2, 0, "%.2f" % cx), (cx, sx / 2.0, "%.2f" % sx)]
        )
        labels = labels.opts(
            padding=0.1, xaxis=None, yaxis=None, text_baseline="bottom"
        )
        return circle * triangle * labels

    vline = hv.DynamicMap(hv.VLine, streams=[tap]).opts(color="black")

    return (plot * vline).opts(toolbar="right")


unit_curve = pn.interact(
    sine, view_fn=hvplot, n=(1, 200), frequency=(0, 10.0)
)

sine_panel = pn.Column(
    pn.Row("# Sinuskurve\ninterkativ veränderbar mit\n1. Frequenz der Sinuskurve\n2. Anzahl der Punkte zum Rendern der Sinuskurve"),
    pn.Row(
        pn.Spacer(width=45),
        unit_curve[0][0],
        unit_curve[0][2],
    ),
    unit_curve[1],
    pn.Row("Bildunterschrift")
)

In [5]:
sine_panel.servable()

Column
    [0] Row
        [0] Markdown(str)
    [1] Row
        [0] Spacer(width=45)
        [1] FloatSlider(end=10.0, label='frequency', name='frequency', value=1.0)
        [2] IntSlider(end=200, label='n', name='n', start=1, value=200)
    [2] Row(sizing_mode='fixed')
        [0] HoloViews(DynamicMap, height=300, name='interactive00257', sizing_mode='fixed', width=500)
    [3] Row
        [0] Markdown(str)

Alternativ könnt ihr den Bokeh-Server auch über die Kommandozeile starten mit

```console
uv run panel serve --show notebooks/panel_dashboard.ipynb
```